## Proyecto - Equipo A

Primero, importamos las librerías necesarias.

In [ ]:
import pandas as pd
import numpy as np
import time
import passwords


from bs4 import BeautifulSoup
import requests

import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from datetime import datetime, timedelta
from time import sleep
import random

import spotipy
from spotipy.oauth2 import SpotifyClientCredentials
from spotipy.oauth2 import SpotifyOAuth

Preparamos los datos necesarios para hacer la request a la web de Billboard y extraer los datos desde noviembre de 2023 hasta hoy. Simplemente, tendremos que añadir como endpoint la fecha a la que queramos acceder para ver la lista en dicho momento.

In [572]:
base_url = "https://www.billboard.com/charts/hot-100/"

# con esto, haremos creer a la web web que no estamos haciendo peticiones masivas y saltarnos el bloqueo
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36',
    'Accept-Language': 'en-US,en;q=0.9',
}

# declaramos las fechas de inicio y fin de la extracción de datos.
start_date = datetime(2023, 11, 16)
end_date = datetime(2024, 11, 16)

# metemos en una lista las fechas que usaremos como endpoint para acceder a los datos
dates = []
current_date = start_date
while current_date <= end_date:
    dates.append(current_date.strftime('%Y-%m-%d'))
    current_date += timedelta(weeks=1)

print(dates)

['2023-11-16', '2023-11-23', '2023-11-30', '2023-12-07', '2023-12-14', '2023-12-21', '2023-12-28', '2024-01-04', '2024-01-11', '2024-01-18', '2024-01-25', '2024-02-01', '2024-02-08', '2024-02-15', '2024-02-22', '2024-02-29', '2024-03-07', '2024-03-14', '2024-03-21', '2024-03-28', '2024-04-04', '2024-04-11', '2024-04-18', '2024-04-25', '2024-05-02', '2024-05-09', '2024-05-16', '2024-05-23', '2024-05-30', '2024-06-06', '2024-06-13', '2024-06-20', '2024-06-27', '2024-07-04', '2024-07-11', '2024-07-18', '2024-07-25', '2024-08-01', '2024-08-08', '2024-08-15', '2024-08-22', '2024-08-29', '2024-09-05', '2024-09-12', '2024-09-19', '2024-09-26', '2024-10-03', '2024-10-10', '2024-10-17', '2024-10-24', '2024-10-31', '2024-11-07', '2024-11-14']


Esta es la función principal para parsear la web. Tomará la URL base, los headers y la lista con las fechas.

In [ ]:
def get_info(url, user_headers, user_date):
    "Obtiene los datos de una lista para la semana de la fecha indicada y lo devuelve en un dataframe"

    # respuesta y objeto BS
    response = requests.get(url, headers=user_headers)
    soup = BeautifulSoup(response.text, "html.parser")

    # listas de las que extraeremos los datos
    # la mayoría los extraemos del mismo objeto soup_elements
    # la posición en la lista y la imagen de la canción debemos extraerla de otros diferentes
    soup_elements = soup.find_all("ul", class_="lrv-a-unstyle-list lrv-u-flex lrv-u-height-100p lrv-u-flex-direction-column@mobile-max")
    soup_position = soup.find_all("li", class_="o-chart-results-list__item // lrv-u-background-color-black lrv-u-color-white u-width-100 u-width-55@mobile-max u-width-55@tablet-only lrv-u-height-100p lrv-u-flex lrv-u-flex-direction-column lrv-u-flex-shrink-0 lrv-u-align-items-center lrv-u-justify-content-center lrv-u-border-b-1 u-border-b-0@mobile-max lrv-u-border-color-grey")
    soup_image = soup.find_all("div", class_="lrv-a-crop-1x1 a-crop-67x100@mobile-max")

    # declaramos las listas vacías donde guardaremoslos datos
    titles = []
    artists = []
    this_week = []
    last_week = []
    peaks = []
    wks_on_chart = []
    images = []
    

    for elem in soup_position:
    # posición actual en la fecha dada
        try:
            this_week.append(elem.find("span").text.replace("\n","").replace("\t",""))
                                                        
        except:
            this_week.append(np.nan)

    for elem in soup_elements:

    # títulos de las canciones
        try:
            titles.append(elem.find("h3", id="title-of-a-story").text.replace("\n","").replace("\t",""))
        except:
            titles.append(np.nan)
    #-----------------------------------------------------------------------------------------------------------------------------------------------
    # artistas
    for elem in soup_elements:
        try:
            artists.append(elem.find("span",).text.replace("\n","").replace("\t",""))
        except:
            artists.append(np.nan)
    #-----------------------------------------------------------------------------------------------------------------------------------------------
    
    #-----------------------------------------------------------------------------------------------------------------------------------------------
    # posición de la semana anterior
        try:
            last_week.append(elem.find_all("li")[3].text.replace("\n","").replace("\t",""))                  
        except:
            last_week.append(np.nan)
    #-----------------------------------------------------------------------------------------------------------------------------------------------
    # posición más alta
        try:
            peaks.append(elem.find_all("li")[4].text.replace("\n","").replace("\t",""))                  
        except:
            peaks.append(np.nan)
    #-----------------------------------------------------------------------------------------------------------------------------------------------
    # semanas en lista
        try:
            wks_on_chart.append(elem.find_all("li")[5].text.replace("\n","").replace("\t",""))                  
        except:
            wks_on_chart.append(np.nan)
    #-----------------------------------------------------------------------------------------------------------------------------------------------
    # imagen
    for elem in soup_image:
        try:
            images.append(elem.find("img")["src"])
        except:
            images.append(np.nan)
    #-----------------------------------------------------------------------------------------------------------------------------------------------
    # fecha de la semana
    week = [user_date for i in range(100)]

    #print(len(titles), len(artists), len(this_week), len(last_week), len(peaks), len(wks_on_chart))   

    # sleep
    sleep(random.uniform(0.5, 2.5))
    
    # guardamos en un df
    df_data = pd.DataFrame({"Title" : titles,
                 "Artist" : artists, 
                 "Pos. this week" : this_week, 
                 "Pos. last week" : last_week, 
                 "Peak position" : peaks, 
                 "Weeks on chart" : wks_on_chart,
                 "Image" : images,
                 "Date" : week})
    
    # print(response.status_code)
    
    return df_data
    
    

Declaramos un dataframe vacío donde guardremos todos los datos, los cuales vamos concatenando, semana a semana. Luego lo guardamos en un csv.

In [ ]:
df = pd.DataFrame(columns=["Title", "Artist", "Pos. this week", "Pos. last week", "Peak position", "Weeks on chart", "Image", "Date"])
for date in dates:
    df = pd.concat([df, get_info(f"{base_url}{date}/", headers, date)], ignore_index=True)

In [ ]:
# df.to_csv("billboard100.csv", index=False)

In [9]:
df = pd.read_csv('billboard100.csv')
df

,Title,Artist,Pos. this week,Pos. last week,Peak position,Weeks on chart,Image,Date
0,Cruel Summer,Taylor Swift,1,6,1,27,https://charts-static.billboard.com/img/2019/0...,2023-11-16
1,Paint The Town Red,Doja Cat,2,4,1,14,https://charts-static.billboard.com/img/2023/0...,2023-11-16
2,Is It Over Now? (Taylor's Version) [From The V...,Taylor Swift,3,1,1,2,https://charts-static.billboard.com/img/2006/0...,2023-11-16
3,Snooze,SZA,4,8,2,48,https://charts-static.billboard.com/img/2022/1...,2023-11-16
4,Standing Next To You,Jung Kook,5,-,5,1,https://charts-static.billboard.com/img/2023/1...,2023-11-16
...,...,...,...,...,...,...,...,...
5295,Hollon,GloRilla,96,83,48,7,https://charts-static.billboard.com/img/2022/0...,2024-11-14
5296,Lonely Road,mgk & Jelly Roll,97,-,33,14,https://charts-static.billboard.com/img/2024/0...,2024-11-14
5297,Change Me,BigXthaPlug,98,100,79,4,https://charts-static.billboard.com/img/2024/1...,2024-11-14
5298,Him All Along,Gunna,99,98,58,3,https://charts-static.billboard.com/img/2024/1...,2024-11-14


Para usar la API de Spotify, necesitamos registrarnos y crear una aplicación en en la web. Esto nos dará unas credenciales para poder acceder a los datos.

In [ ]:
# Replace with your Spotify API credentials
client_id = passwords.id
client_secret = passwords.secret

# Authenticate using Client Credentials Flow
client_credentials_manager = SpotifyClientCredentials(client_id=client_id, client_secret=client_secret)
sp = spotipy.Spotify(client_credentials_manager=client_credentials_manager)

print("Authentication successful!")

Podemos hacer una prueba para ver una canción individual.

In [401]:
# # Search for a track by name
# track_name = "Jingle Bells"
# artist_name = "Frank Sinatra"

def song_data(track_name, artist_name):
    query = f"track:{track_name} artist:{artist_name}"
    results = sp.search(q=query, type="track", limit=1)

    # Extract and print track details
    for track in results['tracks']['items']:
        print(f"Track Name: {track['name']}")
        print(f"Artist: {track['artists'][0]['name']}")
        print(f"Album: {track['album']['name']}")
        print(f"Release date: {track['album']['release_date']}")
        print(f"Popularity: {track['popularity']}")
        print(f"Spotify URL: {track['external_urls']['spotify']}")
        print(f"Song ID: {track['id']}")
        artist_id = track['artists'][0]['id']
        print(f"Artist ID: {artist_id}")
        artist_info = sp.artist(artist_id)
        print(f"Genres: {artist_info['genres']}")
        print(f"Followers: {artist_info['followers']['total']}")

    return


In [402]:
song_data("Jingle Bells", "Frank Sinatra")

Track Name: Jingle Bells - Remastered 1999
Artist: Frank Sinatra
Album: A Jolly Christmas From Frank Sinatra
Release date: 1957-09
Popularity: 71
Spotify URL: https://open.spotify.com/track/4HEOgBHRCExyYVeTyrXsnL
Song ID: 4HEOgBHRCExyYVeTyrXsnL
Artist ID: 1Mxqyy3pSjf8kZZL4QVxS0
Genres: ['adult standards', 'easy listening', 'lounge']
Followers: 7591282


Crearemos un dataframe con los títulos y artistas únicos que podemos encontrar en la lista Billboard para extraer los datos de cada una con la API de Spotify.

In [415]:
songlist = []
artistlist = []
# track_ids = []
for song,artist in zip(df["Title"], df["Artist"]):
    if song not in songlist:
        songlist.append(song)
        artistlist.append(artist)

    else:
        pass

In [416]:
df_metadata = pd.DataFrame()
df_metadata["Title"] = songlist
df_metadata["Artist"] = artistlist

In [7]:
# df_metadata.to_csv("df_metadata.csv", index=False)

La función principal para recoger datos va cogiendo canciones por lotes, ysa que la API tiene sus limitaciones. Además, lotes muy grandes, aunque estén dentro del límite de 100, pueden llegar a dar errores.

Aquí, hemos tenido problemas usando el método <b>sp.artist</b>, necesario para obtener los géneros asociados a cada artista, así como el número de seguidores. Durante esta semana, dejó de fucionar y nos ha sido imposible extraer esos datos.

In [420]:
def fetch_tracks_in_batches(df, sp, batch_size=50):
    """
    Extracción de datos con la API de Spotify por lotes
    Args:
        df (DataFrame): DataFrame con las canciones y artistas que buscaremos.
        sp (Spotify): autenticación.
        batch_size (int): número de canciones que se procesan a la vez.
    Retorna:
        DataFrame con los datos de cada canción.
    """
    all_metadata = []

    for i in range(0, len(df), batch_size):
        batch = df.iloc[i:i + batch_size]
        print(f"Processing batch {i // batch_size + 1}...")

        for _, row in batch.iterrows():
            try:
                
                # Limpieza del formato de los nombres de los artistas para aumentar las probabilidades de matching
                temp_title = row['Title'].replace('The ', "").replace("We ","")   # esto lo hacemos porque si no aparecen duplicados
                temp_artist = row['Artist'].replace("& ", "").replace("And ", "").replace("Featuring ", "").replace("¥$: ", "")
                query = f"track:{temp_title} artist:{temp_artist}"
                results = sp.search(q=query, type="track", limit=1)
                # print(f"data found: {results}")
                # print(f"tracks,items: {results['tracks']['items']}")
                if results['tracks']['items']:

                    track = results['tracks']['items'][0]

                    # print(f"tracks,items,0: {results['tracks']['items'][0]}")

                    # ID del artista para extraer género. 
                    artist_id = [artist['id'] for artist in track['artists']][0]
                    # print(f"artist id: {artist_id}")
                    artist_info = sp.artist(artist_id)
                    # print(f"finding: {artist_id}, {artist_info}")

                    # Extracción de metadatos
                    all_metadata.append({
                        "track_name": row['Title'],
                        "artist_name": row['Artist'],
                        "album_name": track["album"]["name"],
                        "track_id": track['id'],
                        "artist_id": artist_id,
                        "popularity": track['popularity'],
                        "release_date": track['album']['release_date'],
                        "spotify_url": track['external_urls']['spotify'],
                        "genres": artist_info["genres"],  # Get genres
                        "followers": artist_info['followers']['total']  # Get artist followers
                    })
                    # print("data appended")
                else:
                    all_metadata.append({
                        "track_name": row['Title'],
                        "artist_name": row['Artist'],
                        "album_name": None,
                        "track_id": None,
                        "artist_id": None,
                        "popularity": None,
                        "release_date": None,
                        "spotify_url": None,
                        "genres": None,
                        "followers": None
                    })
                    # print("Nones appended")
            except Exception as e:
                print(f"Error processing track {row['Title']} by {row['Artist']}: {e}")
                all_metadata.append({
                    "track_name": row['Title'],
                    "artist_name": row['Artist'],
                    "album_name": None,
                    "track_id": None,
                    "artist_id": None,
                    "popularity": None,
                    "release_date": None,
                    "spotify_url": None,
                    "genres": None,
                    "followers": None
                })
        # Delay
        sleep(random.uniform(0.5, 1.5))
        

    # Retorna un df
    return pd.DataFrame(all_metadata)


Usamos la función y concatenamos los datos en un nuevo df llamado **df_metadata**.

In [418]:
df_tracks = fetch_tracks_in_batches(df_metadata, sp)
df_metadata = pd.concat([df_metadata, df_tracks], axis=1)

Processing batch 1...
Processing batch 2...
Processing batch 3...
Processing batch 4...
Processing batch 5...
Processing batch 6...
Processing batch 7...
Processing batch 8...
Processing batch 9...
Processing batch 10...
Processing batch 11...
Processing batch 12...
Processing batch 13...
Processing batch 14...
Processing batch 15...
Processing batch 16...
Processing batch 17...
Processing batch 18...
Processing batch 19...


In [419]:
df_metadata

,Title,Artist,track_name,artist_name,album_name,track_id,artist_id,popularity,release_date,spotify_url,genres,followers
0,Cruel Summer,Taylor Swift,Cruel Summer,Taylor Swift,Lover,1BxfuPKGuaTgP7aM0Bbdwr,06HL4z0CvFAxyc27GXpf02,89.0,2019-08-23,https://open.spotify.com/track/1BxfuPKGuaTgP7a...,[pop],126410681.0
1,Paint The Town Red,Doja Cat,Paint The Town Red,Doja Cat,Paint The Town Red,2IGMVunIBsBLtEQyoI1Mu7,5cj0lLjcoR7YOSnhnX0Po5,78.0,2023-08-04,https://open.spotify.com/track/2IGMVunIBsBLtEQ...,"[dance pop, pop]",31900940.0
2,Is It Over Now? (Taylor's Version) [From The V...,Taylor Swift,Is It Over Now? (Taylor's Version) [From The V...,Taylor Swift,1989 (Taylor's Version),1Iq8oo9XkmmvCQiGOfORiz,06HL4z0CvFAxyc27GXpf02,75.0,2023-10-26,https://open.spotify.com/track/1Iq8oo9XkmmvCQi...,[pop],126410681.0
3,Snooze,SZA,Snooze,SZA,SOS,4iZ4pt7kvcaH6Yo8UoZ4s2,7tYKF4w9nC0nq9CsPZTHyP,86.0,2022-12-09,https://open.spotify.com/track/4iZ4pt7kvcaH6Yo...,"[pop, r&b, rap]",24431271.0
4,Standing Next To You,Jung Kook,Standing Next To You,Jung Kook,GOLDEN,2KslE17cAJNHTsI2MI0jb2,6HaGTQPmzraVmaVxvz6EUc,89.0,2023-11-03,https://open.spotify.com/track/2KslE17cAJNHTsI...,[k-pop],16483609.0
...,...,...,...,...,...,...,...,...,...,...,...,...
755,Chill Bae,Lil Uzi Vert,Chill Bae,Lil Uzi Vert,Eternal Atake 2,4ppKM7xnkSAwSyKqD4QTY4,4O15NlyKLIASxsJ0PrXPfz,76.0,2024-11-01,https://open.spotify.com/track/4ppKM7xnkSAwSyK...,"[hip hop, melodic rap, philly rap, rage rap, r...",18043364.0
756,Light Year (Practice),Lil Uzi Vert,Light Year (Practice),Lil Uzi Vert,Eternal Atake 2,3NE6g9h77FZbQqCLzUpGRt,4O15NlyKLIASxsJ0PrXPfz,73.0,2024-11-01,https://open.spotify.com/track/3NE6g9h77FZbQqC...,"[hip hop, melodic rap, philly rap, rage rap, r...",18043364.0
757,Sao Paulo,The Weeknd & Anitta,Sao Paulo,The Weeknd & Anitta,São Paulo,0xGQTpN4ESETtl1Sp6rzd2,1Xyo4u8uXC1ZmMpatF05PJ,86.0,2024-10-31,https://open.spotify.com/track/0xGQTpN4ESETtl1...,"[canadian contemporary r&b, canadian pop, pop]",94767820.0
758,We Good,Lil Uzi Vert,We Good,Lil Uzi Vert,Eternal Atake 2,0F60WLPs6Oz2slHomvLt9U,4O15NlyKLIASxsJ0PrXPfz,70.0,2024-11-01,https://open.spotify.com/track/0F60WLPs6Oz2slH...,"[hip hop, melodic rap, philly rap, rage rap, r...",18043364.0


In [421]:
df_metadata.to_csv('dataset_1.csv', index=False)

In [ ]:
# df_metadata = pd.read_csv("dataset_1.csv")

Con esta segunda función, recogemos los parámetros asociados a cada canción, tales como la duración, la bailabilidad, el tempo, etc. Esta necesita el id de cada canción para acceder a ellos.

In [422]:
def fetch_audio_features_in_batches(track_ids, sp, batch_size=40):
    """
    Extraer los parámetros de cada canción.
    Args:
        track_ids (list): Lista con los IDs de las canciones.
        sp (Spotify): Autenticación.
        batch_size (int): número de canciones que se procesan a la vez.
    Retorna:
        lista: con los parámetros de las canciones
    """
    # lista vacía ddonde se guardarán los parámetros
    all_features = []
    
    # Dividir en lotes
    for i in range(0, len(track_ids), batch_size):
        batch = track_ids[i:i + batch_size] 
        
        try:
            # Extraer la info para el batch actual
            features = sp.audio_features(batch)
            
            # Guardar la info en la lista all_features
            if features:
                all_features.extend(features)
            
            # Delay
            time.sleep(random.uniform(0.5, 1.5)) 
            
        except Exception as e:
            print(f"Error fetching batch {i // batch_size + 1}: {e}")
            continue  
    
    return all_features


In [423]:
# IDs de las canciones
track_ids = df_metadata['track_id'].dropna().tolist()  # Eliminamos los Nan

# Extraer los datos
audio_features = fetch_audio_features_in_batches(track_ids, sp)

# Pasamos a df
df_audio_features = pd.DataFrame(audio_features)

# Renombramos la columna de 'id' que nos devuelve, ya que es la misma que 'track_id que tenemos
df_audio_features.rename(columns={'id': 'track_id'}, inplace=True)

# Unimos los dataframe, asegurando que coinciden los datos con los de su canción correspondiente
df_metadata = pd.merge(df_metadata, df_audio_features, on='track_id', how='left')

# Reset index
df_metadata.reset_index(drop=True, inplace=True)


In [424]:
pd.DataFrame(track_ids).duplicated().sum()

np.int64(1)

In [425]:
df_metadata.shape

(762, 29)

In [426]:
df_metadata.to_csv("dataset_2.csv", index=False)

In [ ]:
# df_test = df_metadata

In [427]:
# df_metadata = pd.read_csv("metadata.csv")
df_metadata.columns

Index(['Title', 'Artist', 'track_name', 'artist_name', 'album_name',
       'track_id', 'artist_id', 'popularity', 'release_date', 'spotify_url',
       'genres', 'followers', 'danceability', 'energy', 'key', 'loudness',
       'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness',
       'valence', 'tempo', 'type', 'uri', 'track_href', 'analysis_url',
       'duration_ms', 'time_signature'],
      dtype='object')

In [ ]:
df_metadata.info()

In [ ]:
df_metadata[df_metadata.isna().any(axis=1)]

In [430]:
df_nan = df_metadata[df_metadata.isna().any(axis=1)]
df_nan = df_nan[df_nan["track_id"].notna()]


In [ ]:
pd.set_option("display.max_columns", None)   # para ver todas las columnas del df en el display

In [ ]:
# missing_tracks = df_nan["track_id"].to_list()

In [ ]:
# # Fetch audio features and artist metadata
# audio_features_mt = fetch_audio_features_in_batches(missing_tracks, sp)

# # Convert the audio features to a DataFrame
# df_audio_features_mt = pd.DataFrame(audio_features_mt)

Nos quedamos con las columnas que nos pueden resultar de interés.

In [431]:
df_metadata = df_metadata[['Title', 'Artist', 'album_name',
       'track_id', 'artist_id', 'popularity', 'release_date',
       'genres', 'followers', 'danceability', 'energy', 'key', 'loudness',
       'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness',
       'valence', 'tempo', 'duration_ms']]

In [432]:
df_metadata.tail()

,Title,Artist,album_name,track_id,artist_id,popularity,release_date,genres,followers,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_ms
757,Chill Bae,Lil Uzi Vert,Eternal Atake 2,4ppKM7xnkSAwSyKqD4QTY4,4O15NlyKLIASxsJ0PrXPfz,76.0,2024-11-01,"[hip hop, melodic rap, philly rap, rage rap, r...",18043364.0,0.672,0.663,3.0,-5.984,1.0,0.2430,0.1120,0.0000,0.1340,0.381,184.057,138324.0
758,Light Year (Practice),Lil Uzi Vert,Eternal Atake 2,3NE6g9h77FZbQqCLzUpGRt,4O15NlyKLIASxsJ0PrXPfz,73.0,2024-11-01,"[hip hop, melodic rap, philly rap, rage rap, r...",18043364.0,0.637,0.692,1.0,-7.231,0.0,0.1640,0.0153,0.0000,0.3450,0.427,140.203,164366.0
759,Sao Paulo,The Weeknd & Anitta,São Paulo,0xGQTpN4ESETtl1Sp6rzd2,1Xyo4u8uXC1ZmMpatF05PJ,86.0,2024-10-31,"[canadian contemporary r&b, canadian pop, pop]",94767820.0,0.574,0.659,5.0,-7.377,0.0,0.0768,0.0412,0.0002,0.0921,0.506,127.347,301623.0
760,We Good,Lil Uzi Vert,Eternal Atake 2,0F60WLPs6Oz2slHomvLt9U,4O15NlyKLIASxsJ0PrXPfz,70.0,2024-11-01,"[hip hop, melodic rap, philly rap, rage rap, r...",18043364.0,0.587,0.800,11.0,-3.194,1.0,0.2430,0.0461,0.0000,0.3440,0.823,144.332,125000.0
761,Brokey,Latto,Sugar Honey Iced Tea,1JF8LkQWZ8grXb2emqrNKn,3MdXrJWsbVzdn6fe5JYkSQ,70.0,2024-08-09,[trap queen],1825484.0,0.776,0.525,1.0,-6.723,1.0,0.1370,0.0366,0.0000,0.1080,0.467,75.013,217613.0


Vamos a introducir manualmente alguna de las canciones de las cuales no tenemos los datos. La búsqueda con la API no siempre encuentra bien las canciones. También depende del formato del título y del artista, así como del país desde donde se busque.

In [433]:
df_metadata[df_metadata["Title"] == "Jingle Bells"]

,Title,Artist,album_name,track_id,artist_id,popularity,release_date,genres,followers,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_ms
136,Jingle Bells,Frank Sinatra With The Orchestra & Chorus Of G...,None,None,None,NaN,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [434]:
# Search for a track by name
track_name = "Jingle Bells"
artist_name = "Frank Sinatra"
query = f"track:{track_name} artist:{artist_name}"
results = sp.search(q=query, type="track", limit=1)

# Extract and print track details
for track in results['tracks']['items']:
    print(f"Track Name: {track['name']}")
    print(f"Artist: {track['artists'][0]['name']}")
    print(f"Album: {track['album']['name']}")
    print(f"Release date: {track['album']['release_date']}")
    print(f"Popularity: {track['popularity']}")
    print(f"Spotify URL: {track['external_urls']['spotify']}")
    print(f"Song ID: {track['id']}")
    artist_id = track['artists'][0]['id']
    print(f"Artist ID: {artist_id}")
    artist_info = sp.artist(artist_id)
    print(f"Genres: {artist_info['genres']}")
    print(f"Followers: {artist_info['followers']['total']}")

Track Name: Jingle Bells - Remastered 1999
Artist: Frank Sinatra
Album: A Jolly Christmas From Frank Sinatra
Release date: 1957-09
Popularity: 71
Spotify URL: https://open.spotify.com/track/4HEOgBHRCExyYVeTyrXsnL
Song ID: 4HEOgBHRCExyYVeTyrXsnL
Artist ID: 1Mxqyy3pSjf8kZZL4QVxS0
Genres: ['adult standards', 'easy listening', 'lounge']
Followers: 7591282


In [435]:
vals1 = ['Jingle Bells', 'Frank Sinatra', 'A Jolly Christmas From Frank Sinatra','4HEOgBHRCExyYVeTyrXsnL','1Mxqyy3pSjf8kZZL4QVxS0', 70., '1957-09', [['adult standards', 'easy listening', 'lounge']], 7591282.]
cols1=['Title', 'Artist', 'album_name','track_id', 'artist_id', 'popularity', 'release_date', 'genres', 'followers']
df_sinatra1 = pd.DataFrame(dict(zip(cols1, vals1)))
df_sinatra1

,Title,Artist,album_name,track_id,artist_id,popularity,release_date,genres,followers
0,Jingle Bells,Frank Sinatra,A Jolly Christmas From Frank Sinatra,4HEOgBHRCExyYVeTyrXsnL,1Mxqyy3pSjf8kZZL4QVxS0,70.0,1957-09,"[adult standards, easy listening, lounge]",7591282.0


In [436]:
sinatra = fetch_audio_features_in_batches('4HEOgBHRCExyYVeTyrXsnL',sp)

In [437]:
df_sinatra2 = pd.DataFrame(sinatra)
df_sinatra2

,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,type,id,uri,track_href,analysis_url,duration_ms,time_signature
0,0.511,0.34,8,-13.119,1,0.049,0.727,0,0.0977,0.952,174.661,audio_features,4HEOgBHRCExyYVeTyrXsnL,spotify:track:4HEOgBHRCExyYVeTyrXsnL,https://api.spotify.com/v1/tracks/4HEOgBHRCExy...,https://api.spotify.com/v1/audio-analysis/4HEO...,120693,4


In [438]:
df_sinatra = pd.merge(df_sinatra1, df_sinatra2, left_on='track_id', right_on= 'id')

In [439]:
df_sinatra = df_sinatra[[ 'Title', 'Artist',
       'album_name', 'track_id', 'artist_id','popularity', 'release_date',
       'genres', 'followers', 'danceability', 'energy', 'key', 'loudness',
       'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness',
       'valence', 'tempo', 'duration_ms']]

In [440]:
df_sinatra

,Title,Artist,album_name,track_id,artist_id,popularity,release_date,genres,followers,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_ms
0,Jingle Bells,Frank Sinatra,A Jolly Christmas From Frank Sinatra,4HEOgBHRCExyYVeTyrXsnL,1Mxqyy3pSjf8kZZL4QVxS0,70.0,1957-09,"[adult standards, easy listening, lounge]",7591282.0,0.511,0.34,8,-13.119,1,0.049,0.727,0,0.0977,0.952,174.661,120693


In [441]:
df_sinatra.shape

(1, 21)

In [442]:
df_metadata = pd.concat([df_metadata, df_sinatra], ignore_index=True)

Sinatra añadido.

In [451]:
df_metadata.tail()

,Title,Artist,album_name,track_id,artist_id,popularity,release_date,genres,followers,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_sec,duration_min
758,Light Year (Practice),Lil Uzi Vert,Eternal Atake 2,3NE6g9h77FZbQqCLzUpGRt,4O15NlyKLIASxsJ0PrXPfz,73.0,2024-11-01,"[hip hop, melodic rap, philly rap, rage rap, r...",18043364.0,0.637,0.692,1.0,-7.231,0.0,0.1640,0.0153,0.0000,0.3450,0.427,140.203,164.4,2.74
759,Sao Paulo,The Weeknd & Anitta,São Paulo,0xGQTpN4ESETtl1Sp6rzd2,1Xyo4u8uXC1ZmMpatF05PJ,86.0,2024-10-31,"[canadian contemporary r&b, canadian pop, pop]",94767820.0,0.574,0.659,5.0,-7.377,0.0,0.0768,0.0412,0.0002,0.0921,0.506,127.347,301.6,5.03
760,We Good,Lil Uzi Vert,Eternal Atake 2,0F60WLPs6Oz2slHomvLt9U,4O15NlyKLIASxsJ0PrXPfz,70.0,2024-11-01,"[hip hop, melodic rap, philly rap, rage rap, r...",18043364.0,0.587,0.800,11.0,-3.194,1.0,0.2430,0.0461,0.0000,0.3440,0.823,144.332,125.0,2.08
761,Brokey,Latto,Sugar Honey Iced Tea,1JF8LkQWZ8grXb2emqrNKn,3MdXrJWsbVzdn6fe5JYkSQ,70.0,2024-08-09,[trap queen],1825484.0,0.776,0.525,1.0,-6.723,1.0,0.1370,0.0366,0.0000,0.1080,0.467,75.013,217.6,3.63
762,Jingle Bells,Frank Sinatra,A Jolly Christmas From Frank Sinatra,4HEOgBHRCExyYVeTyrXsnL,1Mxqyy3pSjf8kZZL4QVxS0,70.0,1957-09,"[adult standards, easy listening, lounge]",7591282.0,0.511,0.340,8.0,-13.119,1.0,0.0490,0.7270,0.0000,0.0977,0.952,174.661,120.7,2.01


Vamos a añadir la duración de las canciones en segundos y en minutos.

In [ ]:
df_metadata["duration_ms"] = df_metadata["duration_ms"].apply(lambda x: round(x/1000,1))
df_metadata.rename(columns={"duration_ms": "duration_sec"}, inplace=True)
df_metadata["duration_min"] = round(df_metadata["duration_sec"]/60,2)

In [449]:
df_metadata = df_metadata[df_metadata['track_id'].notna()]

In [450]:
df_metadata[df_metadata['track_id'].duplicated() == True]

,Title,Artist,album_name,track_id,artist_id,popularity,release_date,genres,followers,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_sec,duration_min
322,We Don't Trust You,Future & Metro Boomin,WE STILL DON'T TRUST YOU,7sVbKoBdhXtYCEOO6qC1SN,1RyvyyTE3xzB2ZywiAwp0i,74.0,2024-04-12,"[atl hip hop, hip hop, rap, southern hip hop, ...",19305532.0,0.717,0.586,0.0,-6.15,0.0,0.0273,0.0314,0.26,0.426,0.429,109.993,252.9,4.22
392,We Still Don't Trust You,"Future, Metro Boomin & The Weeknd",WE STILL DON'T TRUST YOU,7sVbKoBdhXtYCEOO6qC1SN,1RyvyyTE3xzB2ZywiAwp0i,74.0,2024-04-12,"[atl hip hop, hip hop, rap, southern hip hop, ...",19305532.0,0.717,0.586,0.0,-6.15,0.0,0.0273,0.0314,0.26,0.426,0.429,109.993,252.9,4.22
393,We Still Don't Trust You,"Future, Metro Boomin & The Weeknd",WE STILL DON'T TRUST YOU,7sVbKoBdhXtYCEOO6qC1SN,1RyvyyTE3xzB2ZywiAwp0i,74.0,2024-04-12,"[atl hip hop, hip hop, rap, southern hip hop, ...",19305532.0,0.717,0.586,0.0,-6.15,0.0,0.0273,0.0314,0.26,0.426,0.429,109.993,252.9,4.22


Quitamos duplicados y vemos qué tamaño tiene el df final.

In [452]:
df_metadata_final = df_metadata.drop_duplicates(subset=["Title"])

In [453]:
df_metadata_final.shape

(742, 22)

Extraemos también el género principal de cada canción.

In [ ]:
genres = df_metadata_final['genres'].tolist()
genre = []
for g in genres:
    if g:
        genre.append(g[0])
    else:
        genre.append("")

df_metadata_final['main_genre'] = genre

lo guardamos en csv.

In [485]:
df_metadata_final.to_csv("metadata_final.csv", index=False)

### Visualizaciones

In [486]:
df_merged = pd.merge(df, df_metadata_final, on=['Title', 'Artist'], how='left')
df_merged

,Title,Artist,Pos. this week,Pos. last week,Peak position,Weeks on chart,Image,Date,album_name,track_id,artist_id,popularity,release_date,genres,followers,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_sec,duration_min,main_genre
0,Cruel Summer,Taylor Swift,1,6,1,27,https://charts-static.billboard.com/img/2019/0...,2023-11-16,Lover,1BxfuPKGuaTgP7aM0Bbdwr,06HL4z0CvFAxyc27GXpf02,89.0,2019-08-23,[pop],126410681.0,0.552,0.702,9.0,-5.707,1.0,0.1570,0.1170,0.000021,0.1050,0.5640,169.994,178.4,2.97,pop
1,Paint The Town Red,Doja Cat,2,4,1,14,https://charts-static.billboard.com/img/2023/0...,2023-11-16,Paint The Town Red,2IGMVunIBsBLtEQyoI1Mu7,5cj0lLjcoR7YOSnhnX0Po5,78.0,2023-08-04,"[dance pop, pop]",31900940.0,0.868,0.538,5.0,-8.603,1.0,0.1740,0.2690,0.000003,0.0901,0.7320,99.968,231.8,3.86,dance pop
2,Is It Over Now? (Taylor's Version) [From The V...,Taylor Swift,3,1,1,2,https://charts-static.billboard.com/img/2006/0...,2023-11-16,1989 (Taylor's Version),1Iq8oo9XkmmvCQiGOfORiz,06HL4z0CvFAxyc27GXpf02,75.0,2023-10-26,[pop],126410681.0,0.599,0.651,0.0,-7.343,1.0,0.0350,0.0486,0.000000,0.1260,0.1530,100.001,229.5,3.82,pop
3,Snooze,SZA,4,8,2,48,https://charts-static.billboard.com/img/2022/1...,2023-11-16,SOS,4iZ4pt7kvcaH6Yo8UoZ4s2,7tYKF4w9nC0nq9CsPZTHyP,86.0,2022-12-09,"[pop, r&b, rap]",24431271.0,0.559,0.551,5.0,-7.231,1.0,0.1320,0.1410,0.000000,0.1100,0.3920,143.008,201.8,3.36,pop
4,Standing Next To You,Jung Kook,5,-,5,1,https://charts-static.billboard.com/img/2023/1...,2023-11-16,GOLDEN,2KslE17cAJNHTsI2MI0jb2,6HaGTQPmzraVmaVxvz6EUc,89.0,2023-11-03,[k-pop],16483609.0,0.711,0.809,2.0,-4.389,0.0,0.0955,0.0447,0.000000,0.3390,0.8160,106.017,206.0,3.43,k-pop
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5295,Hollon,GloRilla,96,83,48,7,https://charts-static.billboard.com/img/2022/0...,2024-11-14,GLORIOUS,1Tt4m8l00BI85uOr5EqXUh,2qoQgPAilErOKCwE2Y8wOG,68.0,2024-10-11,[southern hip hop],1362640.0,0.784,0.755,8.0,-3.352,1.0,0.0583,0.0869,0.000001,0.3120,0.4520,148.910,128.9,2.15,southern hip hop
5296,Lonely Road,mgk & Jelly Roll,97,-,33,14,https://charts-static.billboard.com/img/2024/0...,2024-11-14,Lonely Road (with Jelly Roll),3Rfre3qkrhwdZZ7dyznwbN,6TIYQ3jFPwQSRmorSezPxX,78.0,2024-07-26,"[ohio hip hop, pop rap]",5413811.0,0.549,0.679,7.0,-5.501,1.0,0.0346,0.0103,0.000000,0.3650,0.2350,100.214,189.4,3.16,ohio hip hop
5297,Change Me,BigXthaPlug,98,100,79,4,https://charts-static.billboard.com/img/2024/1...,2024-11-14,TAKE CARE,1AKXrejQiFURJnr9UKIjDb,6qxpnaukVayrQn6ViNvu9I,74.0,2024-10-11,[southern hip hop],1061473.0,0.825,0.648,6.0,-5.122,0.0,0.2720,0.0834,0.000000,0.2030,0.0645,125.009,137.0,2.28,southern hip hop
5298,Him All Along,Gunna,99,98,58,3,https://charts-static.billboard.com/img/2024/1...,2024-11-14,HIM ALL ALONG,2EKL77lwxvo615Ga7VjEUl,2hlmm7s2ICUX0LVIhVFlZQ,76.0,2024-10-18,"[atl hip hop, melodic rap, rap, trap]",8360469.0,0.872,0.507,4.0,-6.323,0.0,0.0720,0.1950,0.000000,0.1110,0.5060,130.007,158.7,2.64,atl hip hop


Vamos a ver la evolución:


In [ ]:
df_top10 = df_merged[df_merged['Pos. this week'] < 11]
# df_top10[df_top10[]].groupby(['Artist','Title'], as_index=False).last()

In [549]:
df_peak = df_merged.groupby(['Title', 'Artist'], as_index=False).agg({'Weeks on chart': 'max'}).rename(columns={'Weeks on chart': 'Max Weeks on Chart'}).sort_values(by='Max Weeks on Chart', ascending=False).reset_index(drop=True)
df_peak_list = df_peak["Title"][0:10].tolist()
df_peak_list
# df_peak[].agg({'Peak position':'max'})

['Snooze',
 'All I Want For Christmas Is You',
 'Lose Control',
 'Last Night',
 "Rockin' Around The Christmas Tree",
 'Fast Car',
 'Jingle Bell Rock',
 'Flowers',
 "Thinkin' Bout Me",
 'Cruel Summer']

In [562]:
df_top10 = df_merged[df_merged['Title'].isin(df_peak_list)]
df_top10['Song (Artist)'] = df_top10['Title'] + " (" + df_top10['Artist'] + ")"

C:\Users\armej\AppData\Local\Temp\ipykernel_6232\2755324081.py:2: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [569]:
fig = px.line(data_frame = df_top10,
       x          = "Date",
       y          = "Pos. this week",
       hover_data = ['Title', 'Artist','Weeks on chart', 'release_date'],
       color      = 'Song (Artist)',
       title='Chart Position of TOP 10 Songs Over Time'
      )

fig.update_layout(
    yaxis=dict(
        autorange="reversed",  # Reverse the order of the y-axis
        title="Chart Position (1 = Top)"),
        title_x=0.5,  # Center the title
        title_y=0.95)

In [584]:
christmas_dates = dates[1:8]
christmas_dates

['2023-11-23',
 '2023-11-30',
 '2023-12-07',
 '2023-12-14',
 '2023-12-21',
 '2023-12-28',
 '2024-01-04']

In [585]:
df_christmas = df_merged[df_merged['Date'].isin(christmas_dates)]
df_top10_ch = df_christmas[df_christmas['Pos. this week'] < 11]

df_top10_ch['Song (Artist)'] = df_top10_ch['Title'] + " (" + df_top10_ch['Artist'] + ")"

C:\Users\armej\AppData\Local\Temp\ipykernel_6232\459495400.py:4: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [593]:
fig = px.line(data_frame = df_top10_ch,
       x          = "Date",
       y          = "Pos. this week",
       hover_data = ['Title', 'Artist','Weeks on chart', 'release_date'],
       color      = 'Song (Artist)',
       title='Chart Position of TOP Songs During Christmas'
      )

fig.update_layout(
    yaxis=dict(
        autorange="reversed",  # Reverse the order of the y-axis
        title="Chart Position (1 = Top)"),
        title_x=0.5,  # Center the title
        title_y=0.95)

In [598]:
dates_24 = dates[24:]
df_2024 = df_merged[df_merged['Date'].isin(dates_24)]
df_top10_24 = df_2024[df_2024['Pos. this week'] < 11]
df_top10_24['Song (Artist)'] = df_top10_24['Title'] + " (" + df_top10_24['Artist'] + ")"

C:\Users\armej\AppData\Local\Temp\ipykernel_6232\3253014253.py:4: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [599]:
fig = px.line(data_frame = df_top10_24,
       x          = "Date",
       y          = "Pos. this week",
       hover_data = ['Title', 'Artist','Weeks on chart', 'release_date'],
       color      = 'Song (Artist)',
       title='Chart Position of TOP Songs Since May 2024'
      )

fig.update_layout(
    yaxis=dict(
        autorange="reversed",  # Reverse the order of the y-axis
        title="Chart Position (1 = Top)"),
        title_x=0.5,  # Center the title
        title_y=0.95)

In [600]:
df_monthly = df_merged[['Date', 'main_genre']]

In [605]:
year_month  = [date[0:7] for date in df_monthly['Date']]


In [608]:
df_monthly['month'] = year_month

C:\Users\armej\AppData\Local\Temp\ipykernel_6232\797323525.py:1: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [692]:
df_filtered = df_monthly[df_monthly['main_genre'] != '']
# df_filtered['month'] = pd.to_datetime(df_filtered['month'])
df_filtered = df_filtered.groupby(['month', 'main_genre'], as_index=False).count().sort_values(by=['month', 'Date'], ascending=False).reset_index(drop=True)

In [693]:
df_filtered['rank'] = df_filtered.groupby('month')['Date'].rank(method='first', ascending=False)

In [694]:
df_top5 = df_filtered[df_filtered['rank'] <= 5]

In [695]:
df_top5

,month,main_genre,Date,rank
0,2024-11,hip hop,31,1.0
1,2024-11,pop,16,2.0
2,2024-11,contemporary country,14,3.0
3,2024-11,southern hip hop,12,4.0
4,2024-11,art pop,10,5.0
...,...,...,...,...
500,2023-11,pop,67,1.0
501,2023-11,contemporary country,44,2.0
502,2023-11,canadian hip hop,22,3.0
503,2023-11,reggaeton,17,4.0


In [ ]:
fig = px.bar(
    df_top5,
    x='month',
    y='Date',
    color='main_genre',
    text='Date',
    title="Top 5 Most Listened Genres by Month",
    labels={'Date': 'Track Count', 'month': 'Month', 'main_genre': 'Genre', 'rank': 'Rank'},
    hover_data={'main_genre': True, 'Date': True, 'rank': True},
    barmode='group',  # To display bars grouped by month
)

# Update layout for better readability
fig.update_layout(
    xaxis_title="Mes",
    yaxis_title="Número de canciones",
    legend_title="Género",
    title_x=0.5,  # Center the title
    hovermode='x unified',
    xaxis=dict(
        tickformat='%Y-%m',  # Format the x-axis ticks as Year-Month
        type='category',     # Treat x-axis as categorical
    )
)

fig.show()

In [784]:
df_genre = df_merged[df_merged['main_genre'] != '']
df_genre = df_genre[['Title', 'Artist', 'main_genre']].reset_index(drop=True)
unique_songs_df = df_genre.drop_duplicates(subset=['Title']).reset_index(drop=True).dropna()
unique_songs_df
# genre_counts = unique_songs_df.groupby('main_genre').size().reset_index(drop=True)
# # genre_counts = unique_songs_df.groupby('main_genre').size().reset_index(name='song_count')

,Title,Artist,main_genre
0,Cruel Summer,Taylor Swift,pop
1,Paint The Town Red,Doja Cat,dance pop
2,Is It Over Now? (Taylor's Version) [From The V...,Taylor Swift,pop
3,Snooze,SZA,pop
4,Standing Next To You,Jung Kook,k-pop
...,...,...,...
717,Chill Bae,Lil Uzi Vert,hip hop
718,Light Year (Practice),Lil Uzi Vert,hip hop
719,Sao Paulo,The Weeknd & Anitta,canadian contemporary r&b
720,We Good,Lil Uzi Vert,hip hop


In [ ]:
fig = px.sunburst(
    unique_songs_df,
    path=['main_genre', 'Artist', 'Title'],  # Define the hierarchy
    title="Music Hierarchy: Genre > Artist > Song",
)

# Customize layout
fig.update_layout(
    title={
        'text': "Music Hierarchy: Genre > Artist > Song",
        'y': 0.9,
        'x': 0.5,
        'xanchor': 'center',
        'yanchor': 'top'
    },
    margin=dict(t=40, l=10, r=10, b=10)  # Adjust margins
)

# Show plot
fig.show()


,Title,Artist,main_genre
0,Cruel Summer,Taylor Swift,pop
1,Paint The Town Red,Doja Cat,dance pop
2,Is It Over Now? (Taylor's Version) [From The V...,Taylor Swift,pop
3,Snooze,SZA,pop
4,Standing Next To You,Jung Kook,k-pop
...,...,...,...
4952,Hollon,GloRilla,southern hip hop
4953,Lonely Road,mgk & Jelly Roll,ohio hip hop
4954,Change Me,BigXthaPlug,southern hip hop
4955,Him All Along,Gunna,atl hip hop


In [761]:
genre_counts

,main_genre,song_count
0,5th gen k-pop,1
1,adult standards,51
2,album rock,4
3,alt z,33
4,alternative metal,7
...,...,...
98,theme,1
99,torch song,4
100,trap queen,43
101,uk alternative hip hop,12


In [ ]:
fig = px.sunburst(data_frame = df_genre_artist,
                #   values     = "loudness",
                  path       = ['main_genre', 'Artist', 'Title'],
                  # hover_name = 'main_genre',
                  # color      = "main_genre"
                  )

# fig.update_traces(textinfo = "label+percent parent")

fig.show()

ValueError: ('None entries cannot have not-None children', main_genre                                     nan
Artist        21 Savage, Young Thug & Metro Boomin
Title                                   Pop Ur Sht
Name: 990, dtype: object)